In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-10-02 10:02:12--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-10-02 10:02:13 (22.6 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [5]:
print("length of dataset in characters: ", len(text))
print(text[:300])

length of dataset in characters:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [9]:
chars = sorted(list(set(text)))
print(''.join(chars))
print(len(chars)) #size of our vocabulary


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [13]:
#we want to tokenize here
#character level tokenization
stoi = {chi:i for i, chi in enumerate(chars)}
itos = {i:chi for i, chi in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
print(encode("shakespeare"))
print(decode(encode("shakespeare")))

[57, 46, 39, 49, 43, 57, 54, 43, 39, 56, 43]
shakespeare


In [20]:
#tokenize the entire dataset we have
#we use pytorch to make the vocab into a tensor
import torch
data =torch.tensor(encode(text))
print(data.shape, data.dtype)
print(data[:50])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])


In [21]:
n = int(0.9*(len(data)))
train = data[:n]
val = data[n:]

In [22]:
#we do not simply feed in entire data to transformer -> compuatationally expensive
#we sample random data, chunks

In [24]:
context_len = 10
train[:context_len+1] #this has 10 examples packed in the sense of -> for these input, this is the char that would come next

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64])

In [26]:
x = train[:context_len]
y = train[1:context_len+1] #offset by 1
for t in range(context_len):
  context = x[:t+1]
  target = y[t]
  print(f"when input is {context} the target: {target}") #transformer should be in the habit of seeing as little as 1 charcater to block_size

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47]) the target: 64


In [ ]:
#mini batches of multiple chunks of text, stacked up, for efficiency to keep GPUs busy
torch.manual_seed(28309123719)
batch_size = 4 #how many chunks are to be processed in parallel
context_len = 10 #characters to be processed for next prediction